# Apply the Customer Engagement Framework to Your Own Data

This notebook is a **practical template** for applying the framework from Xiao and Chen (2025) to a new brand, platform, or industry dataset.

It extends the public quick-start version by demonstrating the full analytical sequence used in the published framework:

1. construct post-level customer engagement (CE) scores using entropy weighting;
2. aggregate post-level records into a brand-period panel;
3. estimate fixed-effect style regression models using EITC variables;
4. simulate and summarize user-generated content (UGC) sentiment results;
5. cluster brands and interpret the results for brand management.

The notebook uses **simulated data by default** so readers can run it immediately. To apply it to your own data, replace the simulated post-level dataset with your own dataset following the schema shown below.

Associated article: Xiao, S., & Chen, X. (2025). *Measuring social media customer engagement with brands based on information entropy: an application case of luxury brand*. Journal of Brand Management, 32, 184–202. https://doi.org/10.1057/s41262-024-00376-7


## 0. Setup

The first cell makes the notebook usable both inside the GitHub repository and directly in Google Colab.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/WindAlan-sw/luxury-brand-customer-engagement.git"
REPO_DIR = Path("luxury-brand-customer-engagement")

# In Colab, clone the repository if the user opened the notebook directly.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and not Path("data").exists() and not REPO_DIR.exists():
    print("Repository files were not found. Cloning the GitHub repository...")
    subprocess.run(["git", "clone", REPO_URL], check=False)

if IN_COLAB and REPO_DIR.exists() and not Path("data").exists():
    os.chdir(REPO_DIR)
    print("Working directory:", Path.cwd())

# Install required packages in Colab if needed.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "matplotlib", "scipy", "statsmodels", "scikit-learn"], check=False)

Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)


## 1. Analytical design and minimum data schema

To apply the framework to another study, your post-level dataset should contain one row per brand-generated post and at least these fields:

| Field | Meaning |
|---|---|
| `brand` | Brand, firm, influencer, or account name |
| `period_idx` | Time-period index, such as month number or campaign period |
| `year_month` | Optional readable period label |
| `retweet_count` | Repost/share behaviour count |
| `reply_count` | Reply/comment behaviour count |
| `like_count` | Like/favourite behaviour count |
| `quote_count` | Quote/repost-with-comment behaviour count |
| `Entertaimment_2` | Entertainment indicator or count for the post |
| `Trendiness_2` | Trendiness indicator or count for the post |
| `Interaction` | Interaction count/indicator for the post |
| `Customization` | Customization/reply-service count/indicator for the post |

The names `Entertaimment_2` and `Trendiness_2` intentionally follow the public metric file used in the repository so the notebook remains consistent with the released fixed-effect panel.


## 2. Simulate a higher-quality example dataset

The previous teaching notebook used a very small toy dataset. This version simulates a larger and more realistic panel:

- 6 brands;
- 72 monthly periods, broadly matching a 2017–2022 style panel;
- about 18,000 brand-generated posts;
- over-dispersed engagement counts;
- brand effects, time effects, and EITC-driven engagement effects.

The simulation is designed to make the regression meaningful: EITC variables are positively related to the latent engagement process, while brand and period effects create realistic heterogeneity.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rng = np.random.default_rng(2025)

brands = ["Armani", "Burberry", "Chanel", "Dior", "Gucci", "LV"]
months = pd.period_range("2017-01", "2022-12", freq="M")

# Brand-level baseline effects: these create stable differences in engagement performance.
brand_effect = {
    "Armani": -0.15,
    "Burberry": -0.05,
    "Chanel": 0.20,
    "Dior": 0.12,
    "Gucci": 0.30,
    "LV": 0.08,
}

# Brand-level communication style tendencies.
style_tendency = {
    "Armani": dict(ent=0.34, trend=0.28, inter=1.15, custom=0.10),
    "Burberry": dict(ent=0.38, trend=0.35, inter=1.25, custom=0.12),
    "Chanel": dict(ent=0.42, trend=0.40, inter=1.35, custom=0.14),
    "Dior": dict(ent=0.40, trend=0.44, inter=1.30, custom=0.15),
    "Gucci": dict(ent=0.46, trend=0.50, inter=1.50, custom=0.16),
    "LV": dict(ent=0.39, trend=0.47, inter=1.40, custom=0.13),
}

rows = []
post_id = 0
for period_idx, ym in enumerate(months, start=1):
    # Period effect captures seasonality and long-term platform change.
    seasonal = 0.12 * np.sin(2 * np.pi * period_idx / 12)
    long_term = 0.004 * period_idx
    for brand in brands:
        # Around 35-50 posts per brand-month, producing a realistic sample size.
        n_posts = int(rng.poisson(42) + 8)
        params = style_tendency[brand]
        for _ in range(n_posts):
            post_id += 1
            entertainment = rng.binomial(1, params["ent"])
            trendiness = rng.binomial(1, params["trend"])
            interaction = 1 + rng.poisson(params["inter"])
            customization = rng.binomial(1, params["custom"])

            # Latent engagement model. Coefficients are chosen so regression can detect effects.
            latent = (
                2.45
                + brand_effect[brand]
                + seasonal
                + long_term
                + 0.18 * entertainment
                + 0.14 * trendiness
                + 0.055 * interaction
                + 0.22 * customization
                + rng.normal(0, 0.28)
            )

            # Generate behavioural counts. Likes are naturally more frequent than replies/quotes.
            base = np.exp(latent)
            like_count = rng.negative_binomial(n=8, p=8 / (8 + base * 4.0))
            retweet_count = rng.negative_binomial(n=6, p=6 / (6 + base * 1.1))
            reply_count = rng.negative_binomial(n=5, p=5 / (5 + base * 0.35))
            quote_count = rng.negative_binomial(n=5, p=5 / (5 + base * 0.22))

            rows.append({
                "post_index": post_id,
                "brand": brand,
                "period_idx": period_idx,
                "year_month": str(ym),
                "Entertaimment_2": entertainment,
                "Trendiness_2": trendiness,
                "Interaction": interaction,
                "Customization": customization,
                "retweet_count": retweet_count,
                "reply_count": reply_count,
                "like_count": like_count,
                "quote_count": quote_count,
            })

posts = pd.DataFrame(rows)
print(posts.shape)
display(posts.head())
print(posts.groupby("brand").size())


## 3. Compute post-level CE scores using entropy weighting

The entropy method assigns larger weights to engagement behaviours that provide more information across posts. This supports the paper's logic of treating different engagement behaviours as unequal information sources rather than simply summing likes, reposts, replies, and quotes.


In [ ]:
def entropy_weights(df: pd.DataFrame, metric_cols: list[str]) -> pd.Series:
    # Calculate Shannon entropy weights for non-negative metric columns.
    X = df[metric_cols].astype(float).clip(lower=0)
    # Add a small constant to avoid log(0) and divide-by-zero problems.
    X = X + 1e-12
    P = X / X.sum(axis=0)
    k = 1 / np.log(len(X))
    entropy = -k * (P * np.log(P)).sum(axis=0)
    diversity = 1 - entropy
    weights = diversity / diversity.sum()
    return weights

metric_cols = ["retweet_count", "reply_count", "like_count", "quote_count"]
weights = entropy_weights(posts, metric_cols)
print("Entropy weights:")
display(weights.to_frame("weight"))

posts["CE_score"] = posts[metric_cols].mul(weights, axis=1).sum(axis=1)
posts["engagement_total"] = posts[metric_cols].sum(axis=1)

display(posts[["brand", "period_idx", "CE_score", "engagement_total"] + metric_cols].head())


### Quick post-level result check

The following table and chart reproduce the first phase of the framework: post-level CE measurement before aggregation. In a real study, this step helps identify which brand-generated contents achieve stronger customer engagement.


In [ ]:
post_summary = posts.groupby("brand").agg(
    post_count=("post_index", "size"),
    mean_CE_score=("CE_score", "mean"),
    median_CE_score=("CE_score", "median"),
    total_CE_score=("CE_score", "sum"),
    total_engagement=("engagement_total", "sum"),
).sort_values("mean_CE_score", ascending=False)

display(post_summary)
post_summary.to_csv("outputs/tables/demo_post_level_brand_summary.csv")

ax = post_summary["mean_CE_score"].plot(kind="bar", figsize=(8, 4), title="Mean post-level CE score by brand")
ax.set_ylabel("Mean CE score")
plt.tight_layout()
plt.savefig("outputs/figures/demo_mean_post_level_ce_score.png", dpi=150)
plt.show()


## 4. Aggregate to a brand-period panel for fixed-effect modelling

The next step follows the published framework more closely: aggregate post-level metrics into brand-period observations. The panel contains period index, brand, sums of EITC variables, CE score sum, and brand dummy variables.

This aggregation mirrors the logic of the fixed-effect model panel used in the paper: brand-generated contents are first scored, then grouped by brand and time period to evaluate how communication dimensions relate to CE outcomes.


In [ ]:
fe_panel = posts.groupby(["brand", "period_idx", "year_month"], as_index=False).agg(
    score=("CE_score", "sum"),
    post_count=("post_index", "size"),
    Entertaimment_2=("Entertaimment_2", "sum"),
    Interaction=("Interaction", "sum"),
    Trendiness_2=("Trendiness_2", "sum"),
    Customization=("Customization", "sum"),
    engagement_total=("engagement_total", "sum"),
)

# Brand dummy variables. Armani is later used as the reference category in regression.
brand_dummies = pd.get_dummies(fe_panel["brand"], prefix="brand", dtype=int)
fe_panel = pd.concat([fe_panel, brand_dummies], axis=1)

print(fe_panel.shape)
display(fe_panel.head())
fe_panel.to_csv("outputs/tables/demo_fixed_effects_model_panel.csv", index=False)


## 5. Regression analysis

This section estimates two models:

- **Model 1**: brand dummy variables only, matching the simpler public fixed-effect panel logic;
- **Model 2**: brand and period fixed effects, controlling for time-period shocks.

In your own study, the choice between pooled OLS, dummy-variable fixed effects, and more formal panel estimators should be justified based on your research design and diagnostics. The purpose here is to automate the core regression workflow aligned with the published framework.


In [ ]:
import statsmodels.formula.api as smf

# Omit brand_Armani as the reference category to avoid perfect multicollinearity with the intercept.
brand_terms = "brand_Burberry + brand_Chanel + brand_Dior + brand_Gucci + brand_LV"
eitc_terms = "Entertaimment_2 + Interaction + Trendiness_2 + Customization"

model_1 = smf.ols(
    f"score ~ {eitc_terms} + {brand_terms}",
    data=fe_panel,
).fit(cov_type="HC3")

model_2 = smf.ols(
    f"score ~ {eitc_terms} + {brand_terms} + C(period_idx)",
    data=fe_panel,
).fit(cov_type="HC3")

print("MODEL 1: EITC variables + brand dummies")
print(model_1.summary())

print("\n\nMODEL 2: EITC variables + brand dummies + period fixed effects")
print(model_2.summary())


### Regression interpretation guide

Interpret the regression in the same spirit as the published paper:

- A positive and significant EITC coefficient suggests that the corresponding communication activity is associated with stronger CE outcomes.
- Brand dummy coefficients compare each brand with the omitted reference brand, here Armani.
- Period fixed effects absorb time-specific changes such as seasonality, platform change, campaign cycles, or macro shocks.
- The regression should not be interpreted as causal unless the research design justifies causal identification.

In engagement marketing terms, statistically meaningful EITC effects suggest that brand-initiated communication activities can shape the intensity of customer interaction. Managerially, this helps identify whether entertainment, trendiness, interaction, or customization should be emphasized in future content strategy.


In [ ]:
def tidy_model(model, name: str) -> pd.DataFrame:
    out = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "p_value": model.pvalues.values,
    })
    out["model"] = name
    return out[["model", "term", "coef", "std_err", "p_value"]]

regression_results = pd.concat([
    tidy_model(model_1, "Model 1: brand dummies"),
    tidy_model(model_2, "Model 2: brand + period fixed effects"),
], ignore_index=True)

regression_results.to_csv("outputs/tables/demo_regression_results.csv", index=False)
display(regression_results[regression_results["term"].isin(["Entertaimment_2", "Interaction", "Trendiness_2", "Customization"])])


## 6. Simulate UGC sentiment results

The published framework combines behavioural engagement from brand-generated content with affective engagement from customer-generated content. Because this teaching notebook should not require raw user text, we simulate UGC sentiment outputs directly.

For your own research, this table could come from a sentiment classifier applied to customer posts, reviews, comments, or other user-generated content. The public version should normally store only aggregated sentiment metrics, not raw user text.


In [ ]:
# Simulate customer-generated content sentiment at brand-month level.
ugc_rows = []
brand_sentiment_base = {
    "Armani": -0.04,
    "Burberry": 0.00,
    "Chanel": 0.10,
    "Dior": 0.08,
    "Gucci": 0.13,
    "LV": 0.06,
}

# Link sentiment weakly to standardized CE performance, as affective engagement may co-move with behavioural engagement.
tmp = fe_panel[["brand", "period_idx", "year_month", "score"]].copy()
tmp["score_z"] = (tmp["score"] - tmp["score"].mean()) / tmp["score"].std()

for row in tmp.itertuples(index=False):
    brand = row.brand
    n_ugc = int(rng.poisson(55 + 8 * max(row.score_z, -2)) + 10)
    mean_sent = np.tanh(brand_sentiment_base[brand] + 0.08 * row.score_z + rng.normal(0, 0.08))
    # Convert mean sentiment tendency into positive/neutral/negative probabilities.
    p_pos = np.clip(0.36 + 0.22 * mean_sent, 0.08, 0.80)
    p_neg = np.clip(0.24 - 0.18 * mean_sent, 0.05, 0.70)
    p_neu = max(0.02, 1 - p_pos - p_neg)
    probs = np.array([p_neg, p_neu, p_pos])
    probs = probs / probs.sum()
    counts = rng.multinomial(n_ugc, probs)
    sentiment_score = (-1 * counts[0] + 0 * counts[1] + 1 * counts[2]) / n_ugc
    ugc_rows.append({
        "brand": brand,
        "period_idx": row.period_idx,
        "year_month": row.year_month,
        "ugc_post_count": n_ugc,
        "negative_count": counts[0],
        "neutral_count": counts[1],
        "positive_count": counts[2],
        "mean_sentiment_score": sentiment_score,
        "positive_share": counts[2] / n_ugc,
        "negative_share": counts[0] / n_ugc,
    })

sentiment_panel = pd.DataFrame(ugc_rows)
display(sentiment_panel.head())
sentiment_panel.to_csv("outputs/tables/demo_ugc_sentiment_panel.csv", index=False)


### Sentiment interpretation guide

Sentiment results should be interpreted as affective engagement indicators. In the published framework, sentiment analysis complements behavioural CE scores because engagement is not only about interaction volume; it also concerns how customers evaluate or emotionally respond to brands.

For brand management, the most useful comparisons are often:

- high CE and positive sentiment: strong engagement with favourable affect;
- high CE and negative sentiment: active but potentially problematic engagement;
- low CE and positive sentiment: favourable but less activated customer base;
- low CE and negative sentiment: weak engagement and possible reputational concern.


In [ ]:
sentiment_summary = sentiment_panel.groupby("brand").agg(
    total_ugc_posts=("ugc_post_count", "sum"),
    mean_sentiment_score=("mean_sentiment_score", "mean"),
    mean_positive_share=("positive_share", "mean"),
    mean_negative_share=("negative_share", "mean"),
).sort_values("mean_sentiment_score", ascending=False)

display(sentiment_summary)
sentiment_summary.to_csv("outputs/tables/demo_brand_sentiment_summary.csv")

ax = sentiment_summary["mean_sentiment_score"].plot(kind="bar", figsize=(8, 4), title="Mean UGC sentiment by brand")
ax.set_ylabel("Mean sentiment score")
plt.tight_layout()
plt.savefig("outputs/figures/demo_mean_ugc_sentiment.png", dpi=150)
plt.show()


## 7. Brand clustering analysis

The clustering step turns multiple engagement indicators into an interpretable brand typology. This follows the spirit of the published paper's hierarchical clustering analysis: brands can be benchmarked not only by a single score, but by patterns across CE, EITC activities, and sentiment.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

brand_features = fe_panel.groupby("brand").agg(
    mean_CE_score=("score", "mean"),
    total_CE_score=("score", "sum"),
    mean_post_count=("post_count", "mean"),
    entertainment_sum=("Entertaimment_2", "sum"),
    interaction_sum=("Interaction", "sum"),
    trendiness_sum=("Trendiness_2", "sum"),
    customization_sum=("Customization", "sum"),
).join(sentiment_summary[["mean_sentiment_score", "mean_positive_share"]])

feature_cols = [
    "mean_CE_score",
    "total_CE_score",
    "mean_post_count",
    "entertainment_sum",
    "interaction_sum",
    "trendiness_sum",
    "customization_sum",
    "mean_sentiment_score",
    "mean_positive_share",
]

Z_input = StandardScaler().fit_transform(brand_features[feature_cols])
linkage_matrix = linkage(Z_input, method="ward")

plt.figure(figsize=(8, 4))
dendrogram(linkage_matrix, labels=brand_features.index.tolist())
plt.title("Hierarchical clustering of simulated brands")
plt.ylabel("Ward distance")
plt.tight_layout()
plt.savefig("outputs/figures/demo_brand_clustering_dendrogram.png", dpi=150)
plt.show()

clusterer = AgglomerativeClustering(n_clusters=3, linkage="ward")
brand_features["cluster"] = clusterer.fit_predict(Z_input) + 1

display(brand_features.sort_values("cluster"))
brand_features.to_csv("outputs/tables/demo_brand_clustering_features.csv")


### Clustering interpretation guide and brand management implications

When interpreting clusters, avoid treating cluster numbers as rankings by themselves. Instead, inspect the features that define each group:

- **High CE / high sentiment brands** may represent strong engagement performers. Their practices can be examined as benchmarks.
- **High CE / lower sentiment brands** may generate attention but require reputational or message-quality diagnosis.
- **Low CE / high sentiment brands** may have goodwill but need stronger activation strategies.
- **Low CE / low sentiment brands** may require broader repositioning of content strategy and customer interaction mechanisms.

Aligned with engagement marketing theory and Xiao and Chen's framework, the managerial value of clustering is not simply naming winners and losers. It helps managers identify which engagement mechanisms should be strengthened: entertainment, interaction, trendiness, customization, or affective customer response.


## 8. Replace the simulated data with your own data

To apply the framework, replace `posts` with your own post-level dataset. The cell below validates that the required columns are available.


In [ ]:
REQUIRED_COLUMNS = {
    "brand",
    "period_idx",
    "retweet_count",
    "reply_count",
    "like_count",
    "quote_count",
    "Entertaimment_2",
    "Interaction",
    "Trendiness_2",
    "Customization",
}

def validate_post_level_data(df: pd.DataFrame) -> None:
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    numeric_cols = list(REQUIRED_COLUMNS - {"brand"})
    non_numeric = [c for c in numeric_cols if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        raise TypeError(f"These columns should be numeric: {non_numeric}")
    if df["brand"].nunique() < 2:
        raise ValueError("At least two brands/groups are recommended for benchmarking and dummy-variable modelling.")
    if len(df) < 500:
        print("Warning: the dataset is small. Regression and clustering results may be unstable.")
    print("Data validation passed.")

validate_post_level_data(posts)


### Optional upload pattern

If you have your own de-texted CSV, uncomment and adapt the following code:

```python
from google.colab import files
uploaded = files.upload()
my_file = next(iter(uploaded.keys()))
posts = pd.read_csv(my_file)
validate_post_level_data(posts)
```

Your public research release should normally **exclude raw text, user handles, URLs, and user-identifying fields**. Keep only the de-texted metrics needed for the framework.


## 9. Summary

This notebook provides a reusable version of the published framework:

1. post-level CE score construction;
2. brand-period aggregation;
3. fixed-effect style regression;
4. UGC sentiment summary;
5. brand clustering and managerial interpretation.

For a real project, report the data source, period, sample size, filtering rules, EITC coding rules, sentiment method, model specification, and limitations. The framework is useful for benchmarking brands and diagnosing which engagement activities are associated with stronger customer responses.
